# Pathology-only grader — seeds 42 → 123 → 2026, then fixed late fusion with P

**Question.** Can the segmentation outputs (Stage-3 vessel map + Stage-4 v2 lesion maps) support DR grading on their own,
and does their information add anything when combined with the stored RGB grader P?

**Model.** vessel + MA / HE / EX / SE maps → Stage-5 pathology encoder → GAP → LayerNorm → CORN. No RGB input, no ConvNeXt.
**Protocol.** The P protocol (batch 2, AdamW 1e-4 / wd 0.05, weighted CORN, ≤ 50 epochs, val-QWK early stopping), P's epoch order
and spatial augmentation, the same bundle, split and seeds. No EMA, no soft targets.

**Fusion — fixed before training, never tuned.** fused_k = ½·P(grade > k)[P, same seed, BEST] + ½·P(grade > k)[pathology, same seed, BEST];
grade = number of k with fused_k > 0.5. Control: the same rule on two P seeds. The P runs are frozen and are not retrained.

GPU runtime. **Cell 4 trains** the three seeds one after another (resumable: after a disconnect rerun cells 1–4).
Cell 5 prints the results. Nothing here defines a success threshold; results are read with the matched-seed framework (record §54).

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_train as at
import pathology_grader_model as pm, pathology_grader_train as pt, pathology_grader_fusion as pf, arch1_posthoc as ph
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: Stage-4 model identity, bundle, protocol, the fixed fusion rule, the stored P runs ====
EXP = colab_config.DRIVE.experiments_root
EXPECTED_MODEL_SHA = "cb5fc7a8d370af7d2ae191cadaebdde858f71820d147fb76f852b757766f8ad8"   # gate-passed Stage-4 v2 (§47/§48)
STAGE4_RUN_DIR = posixpath.join(EXP, "Stage4V2", "stage4v2_k4_seed42_bs8")
MODEL_SHA = json.load(open(posixpath.join(STAGE4_RUN_DIR, "exported_model.json")))["model_sha256"]
assert MODEL_SHA == EXPECTED_MODEL_SHA, f"exported Stage-4 model {MODEL_SHA} is not the approved {EXPECTED_MODEL_SHA}"
GEN4 = cache.stage4_generation_id(MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
assert GEN4 == "s4v2-cb5fc7a8d370-K4", GEN4
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
SIX_RUN_DIR = msr.experiment_root(posixpath.join(EXP, "ImprovedTraining"), "improved_multiseed_2026_09")
PREREG, _ = msr.load_and_verify_preregistration(posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
CLASS_WEIGHTS = list(PREREG["class_weights"])
at.assert_protocol(PREREG, CLASS_WEIGHTS)
SEEDS = pt.SEEDS
assert SEEDS == (42, 123, 2026)
# The fusion rule is fixed here, before anything is trained, and goes into every run's config.json.
assert pf.FUSION["weights"] == [0.5, 0.5] and pf.FUSION["decode_threshold"] == 0.5 and pf.FUSION["tuned_on_validation"] is False
P_TABLES = pt.load_p_tables(EXP)          # the stored P BEST predictions (frozen; never retrained)
RUN_DIRS = {s: pt.run_dir_for(EXP, MODEL_SHA, s) for s in SEEDS}
SUMMARY_DIR = pt.summary_dir_for(EXP, MODEL_SHA)
SOURCES = {"repo_commit": REPO_COMMIT, **{f"p_{s}": pf.p_prediction_path(EXP, s) for s in SEEDS},
           **{f"p_{s}_sha256": cache.sha256_file(pf.p_prediction_path(EXP, s)) for s in SEEDS}}
print("model", MODEL_SHA, "| bundle", BUNDLE_ID)
print("fusion:", pf.FUSION["rule"], "| weights", pf.FUSION["weights"])
for _s in SEEDS:
    print(f"  seed {_s}: {pt.seed_state(RUN_DIRS[_s]):<11} {RUN_DIRS[_s]} | control: P-{pf.control_seed(_s)}")

In [ ]:
# ==== [3] VERIFY THE BUNDLE ON DRIVE, STAGE THE VESSEL + LESION CACHES LOCALLY (~3.8 GB; no RGB), VERIFY, PRE-FLIGHT ====
DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2_pathology", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=MODEL_SHA)
pt.stage_inputs(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=MODEL_SHA, roots=LOCAL_ROOTS)
pt.require_inputs(BUNDLE, BUNDLE.population)
assert BUNDLE.stage3_sha256 == v2cfg.STAGE3_LWNET_SHA256 and BUNDLE.stage4_sha256 == EXPECTED_MODEL_SHA
assert BUNDLE.stage4_generation == GEN4 and BUNDLE.bundle["stage3_generation"] == v2cfg.STAGE3_GENERATION
_tr, _va, _split_sha = msr.verify_split()
_empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
assert _split_sha == v2cfg.SPLIT_SHA256
assert list(BUNDLE.train_ids) == [str(i) for i, _ in _tr if str(i) not in _empty] and len(BUNDLE.train_ids) == 2921
assert list(BUNDLE.val_ids) == [str(i) for i, _ in _va if str(i) not in _empty] and len(BUNDLE.val_ids) == 730
for _s in SEEDS:                           # the stored P predictions are the same validation images, same order
    assert list(P_TABLES[_s]["ids"]) == list(BUNDLE.val_ids), f"P-{_s} was evaluated on a different validation set"
    assert np.array_equal(pf.decode(P_TABLES[_s]["p_gt"]), P_TABLES[_s]["pred"]), f"P-{_s}: stored grades are not the decode rule"
assert not os.path.isdir(BUNDLE.dirs["stage2_rgb_v2"]), "the RGB cache must not be staged for this branch"
# Pre-flight: the model has only the two segmentation inputs, and one real batch runs end to end.
model = pt.build_compiled_model(BUNDLE.channels, SEEDS[0], CLASS_WEIGHTS, mixed_precision=False)
assert [i.name for i in model.inputs] == ["vessel", "pathology"] and not [l.name for l in model.layers if "convnext" in l.name]
_x, _y = pt.make_epoch_sequence(BUNDLE, list(zip(BUNDLE.val_ids[:2], [0, 0])), 0, SEEDS[0], 2, augment=False)[0]
assert set(_x) == {"vessel", "pathology"} and np.isfinite(model.predict_on_batch(_x)).all()
print(json.dumps(BUNDLE.describe(), indent=1))
print("inputs:", pm.input_channel_names(BUNDLE.channels), "| parameters", pm.parameter_report(model))
print("P-only validation QWK:", {s: round(ph.metrics(P_TABLES[s])["qwk"], 4) for s in SEEDS})
del model

In [ ]:
# ==== [4] TRAIN + EVALUATE SEEDS 42 -> 123 -> 2026 (resumable) -- THIS TRAINS ====
# Per seed: fresh pathology grader -> train -> BEST / LAST evaluation -> vessel / per-lesion-class / all-lesion shuffles
# -> fixed fusion with the stored P of the same seed -> P + P control -> result.json / result.md in that seed's directory.
# Any error stops the sequence (no seed is skipped). Rerun cells 1-4 to continue: completed seeds are kept, the
# interrupted seed resumes from its own checkpoint. The 3-seed summary is written only after all three.
REPORTS, SUMMARY = pt.run_sequence(EXP, BUNDLE, CLASS_WEIGHTS, repo_dir=REPO_DIR, staging_root="/content/pathgrader_staging",
                                   sources=SOURCES, p_tables=P_TABLES)

In [ ]:
# ==== [5] REPORT: 3-seed summary + per-seed results (reads what cell 4 wrote; trains nothing) ====
assert all(pt.seed_state(RUN_DIRS[s]) == "complete" for s in SEEDS), {s: pt.seed_state(RUN_DIRS[s]) for s in SEEDS}
assert os.path.exists(posixpath.join(SUMMARY_DIR, "summary.json")), "summary not written -- rerun cell 4"
print(open(posixpath.join(SUMMARY_DIR, "summary.md"), encoding="utf-8").read())
for _s in SEEDS:
    print(open(posixpath.join(RUN_DIRS[_s], "result.md"), encoding="utf-8").read())
print("summary:", SUMMARY_DIR)